# Bronze Layer Build Journal

Este notebook documenta o processo realizado até aqui na construção da camada `bronze/raw data` da plataforma de dados financeiros.

O foco deste material é registrar:
- o contexto do projeto
- as fontes escolhidas
- as decisões de modelagem
- a estrutura do código implementado
- o resultado da primeira execução da ingestão raw


## 1. Contexto do projeto

O projeto foi concebido como uma plataforma de dados financeiros com arquitetura lakehouse em AWS.

Objetivo principal:
- coletar dados financeiros globais
- armazenar dados brutos em uma bronze layer
- preparar dados para transformações posteriores com PySpark
- disponibilizar datasets analíticos para camadas silver, gold e consumo em BI

Arquitetura conceitual definida:

```text
Data Sources
    -> Python Ingestion
    -> S3 Bronze (Raw)
    -> PySpark / Glue / EMR
    -> S3 Silver
    -> S3 Gold / Iceberg
    -> Athena / BI
```

O diagrama visual inicial do projeto foi mantido em `draw/arquitetura.xml`.

## 2. Fontes escolhidas para a V1

As duas fontes escolhidas para a V1 da bronze foram:

- `Yahoo Finance`, usando `yfinance`
- `Alpha Vantage`, usando os endpoints gratuitos

Racional da escolha:

- Yahoo Finance fornece boa cobertura para preços diários, dividendos, splits e metadados de ativos
- Alpha Vantage fornece boa cobertura gratuita para fundamentals, FX, macroeconomia e preços diários simples

Tradeoff importante encontrado na exploração:

- `TIME_SERIES_DAILY_ADJUSTED` da Alpha Vantage não entrou na V1 porque apareceu como endpoint premium
- por isso, a V1 adotou `TIME_SERIES_DAILY` na Alpha Vantage e deixou dividendos/splits principalmente com Yahoo Finance

## 3. Fase de exploração dos dados

Antes de implementar a ingestão produtizada, foi feita uma exploração em notebooks para entender os payloads de cada fonte.

Notebooks criados:

- `01_yahoo_finance_exploration.ipynb`
- `02_alpha_vantage_exploration.ipynb`

Essa etapa serviu para responder perguntas como:

- quais datasets valem a pena para a bronze layer
- quais campos aparecem de forma consistente
- qual o melhor particionamento dos arquivos raw
- quais datasets são mais adequados para dimensões futuras
- quais limitações existem nas APIs gratuitas


## 4. Decisões de modelagem da bronze layer

A principal decisão de modelagem foi:

- a bronze deve preservar o dado bruto o mais próximo possível da resposta original

Ou seja, nesta camada não tentamos impor um schema tabular analítico completo. Em vez disso, cada arquivo raw guarda:

- metadados de ingestão
- informações sobre a requisição
- chaves naturais do payload
- o `raw_payload` retornado pela fonte

Motivações:

- facilitar replay e reprocessamento
- preservar rastreabilidade por fonte
- suportar auditoria e debugging
- deixar a normalização forte para a silver layer

## 5. Datasets definidos para a V1 raw

Yahoo Finance:

- `market_prices_daily`
- `corporate_actions`
- `company_metadata`

Alpha Vantage:

- `stock_prices_daily`
- `company_overview`
- `income_statement`
- `balance_sheet`
- `fx_daily`
- `macro_real_gdp`

Esses datasets foram documentados formalmente em `docs/raw_data_model_v1.md`.

## 6. Estratégia de particionamento

A estrutura definida para os arquivos raw segue um padrão orientado a data lake:

```text
data/bronze/
  source=<source>/
    dataset=<dataset>/
      <natural_key>=<value>/
        ingestion_date=<yyyy-mm-dd>/
          <file>.json
```

Exemplos reais da primeira execução:

- `source=yahoo_finance/dataset=market_prices_daily/symbol=AAPL/...`
- `source=alpha_vantage/dataset=stock_prices_daily/symbol=IBM/...`
- `source=alpha_vantage/dataset=fx_daily/from_symbol=USD/to_symbol=BRL/...`

Esse desenho foi escolhido para facilitar:

- leitura incremental
- reprocessamento por dataset
- rastreamento por chave natural
- evolução futura para S3

## 7. Envelope padrão de ingestão

Cada arquivo raw gerado segue um envelope parecido com este:

```json
{
  "source": "alpha_vantage",
  "dataset": "stock_prices_daily",
  "ingestion_ts_utc": "...",
  "ingestion_date": "...",
  "extraction_mode": "full_refresh",
  "request_params": {...},
  "request_url": "...",
  "request_status_code": 200,
  "natural_keys": {...},
  "api_message_type": null,
  "api_message": null,
  "raw_payload": {...}
}
```

Isso padroniza a camada raw e evita que cada fonte seja tratada de forma completamente diferente.

## 8. Estrutura de código criada

A primeira base produtizável da bronze layer foi implementada com os seguintes componentes:

- `src/contracts/raw_datasets.py`
- `src/storage/raw_writer.py`
- `src/clients/yahoo_finance.py`
- `src/clients/alpha_vantage.py`
- `src/ingestion/run_bronze_ingestion.py`

Papéis de cada módulo:

- `contracts`: define os datasets raw e seus metadados lógicos
- `storage`: grava os envelopes no padrão da bronze
- `clients`: encapsula acesso às APIs externas
- `ingestion`: coordena a execução ponta a ponta

## 9. Ajustes de segurança realizados

Durante o desenvolvimento, foi identificado risco de exposição da API key da Alpha Vantage.

Ajustes aplicados:

- criação do arquivo `.env`
- criação do arquivo `.env.example`
- criação do `.gitignore`
- remoção de hardcode da key no notebook
- mascaramento da key no campo `request_url`

Isso foi importante porque os arquivos raw podem guardar a URL da requisição, e essa URL não deve expor credenciais reais.

## 10. Primeira execução da bronze layer

A primeira execução completa do pipeline raw gerou arquivos em `data/bronze/` com sucesso.

Arquivos de exemplo criados:

- Yahoo Finance `market_prices_daily` para `AAPL`, `MSFT` e `PETR4.SA`
- Yahoo Finance `corporate_actions` para os mesmos tickers
- Yahoo Finance `company_metadata`
- Alpha Vantage `stock_prices_daily` para `IBM`
- Alpha Vantage `company_overview`
- Alpha Vantage `income_statement`
- Alpha Vantage `balance_sheet`
- Alpha Vantage `fx_daily` para `USD/BRL`
- Alpha Vantage `macro_real_gdp`

Esse resultado validou:

- o fluxo ponta a ponta de ingestão
- o formato dos envelopes raw
- a convenção de particionamento
- a viabilidade da V1 da bronze layer

## 11. Estado atual do projeto

Até este ponto, já temos:

- exploração inicial das fontes
- definição da modelagem raw V1
- pipeline local de ingestão raw
- arquivos reais gravados em `data/bronze/`
- notebook de documentação do processo

Em outras palavras: a bronze layer já saiu do campo conceitual e entrou em execução real.

## 12. Próximos passos planejados

Os próximos passos mais naturais a partir daqui são:

- realizar novas execuções para aumentar a massa da bronze
- criar normalizadores da silver layer
- consolidar referências de empresa em datasets canônicos
- definir checagens de qualidade dos dados
- preparar orquestração futura

A transição natural do projeto agora é:

```text
bronze raw funcionando
    -> silver normalizada
    -> gold analítica
```

Este notebook marca o fechamento da fase de fundação da camada bronze.